# Cropy Qwen3.5 기능 토큰 Colab 파일럿 v11
작성자 김헌영 · 구성일 2026-10-08

현재 사람 승인 데이터는 0개입니다. 이 노트북은 승인 검사를 통과하기 전 모델을 다운로드하거나 학습하지 않습니다. **Colab GPU 실행은 아직 검증하지 않았습니다.**

Drive의 `cropy_qwen_v11` 폴더에 수정본 v11, 사람검수표 v11, `v11_사람검수_대조용정답.csv`를 넣고 이 노트북과 `cropy_qwen_pilot.py`를 그 아래 `Colab_파일럿_v11`에 둡니다. 사람 검수표의 독립 판정을 먼저 기록한 뒤 별도 정답과 대조하고 최종 `approved`를 기록하세요.

기본 모델은 검토 후보 `Qwen/Qwen3.5-2B`입니다. 모델 크기를 확인하고 설정을 바꾸면 실제 commit을 조회해 모든 로딩에 고정합니다. 사람 승인과 GPU 설정 확인 후 `RUN_MODEL_STEPS=True`로 바꾸고 위에서부터 실행하세요. 기능별 최소 학습 1건·검증 1건이 필요하며 기존 split을 유지합니다. 이 파일럿은 기능당 400개 본학습이나 실제 handler·TTS 실행이 아닙니다.

In [ ]:
from pathlib import Path
import sys,json,hashlib
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR=Path('/content/drive/MyDrive/cropy_qwen_v11')
CODE_DIR=DATA_DIR/'Colab_파일럿_v11'
SOURCE=DATA_DIR/'Cropy_Qwen35_기능별400_검수대기_v11.xlsx'
HUMAN=DATA_DIR/'Cropy_Qwen35_사람검수표_v11.xlsx'
REFERENCE=DATA_DIR/'v11_사람검수_대조용정답.csv'
MODEL_ID='Qwen/Qwen3.5-2B'  # 점검 후보 크기를 확인하고 수정 가능
MODEL_REF='main'  # 다음 단계에서 실제 commit SHA로 해석 후 고정
RUN_MODEL_STEPS=False
MAX_LENGTH=4096
MAX_STEPS=20
MAX_TRAIN_PER_FUNCTION=16
MAX_EVAL_PER_FUNCTION=4
OUTPUT_DIR=DATA_DIR/'pilot_outputs_v11'
SEED=10810
APPROVAL_READY=TOKEN_CHECKED=MASK_CHECKED=GRADIENT_CHECKED=False
for p in [SOURCE,HUMAN,REFERENCE,CODE_DIR/'cropy_qwen_pilot.py']:
    assert p.is_file(),f'필요한 파일 없음 {p}'


## 1 데이터 승인 검사
원본 SHA와 검수 입력을 대조합니다. 작성 AI의 reviewed 기록은 사람 승인으로 사용하지 않습니다. 승인 수량이 0개이거나 7개 기능 중 학습·검증 사례가 빠지면 여기서 중단합니다.

In [ ]:
import importlib.util,subprocess
if importlib.util.find_spec('openpyxl') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','openpyxl'])
sys.path.insert(0,str(CODE_DIR))
import cropy_qwen_pilot as helper
APPROVAL_READY=False
approved,schemas,tokens,system=helper.load_reviewed_dataset(SOURCE,HUMAN,REFERENCE)
from collections import Counter
print('승인 사례',len(approved),Counter((r['function_name'],r['split']) for r in approved))
train_rows=[];eval_rows=[]
for fn in sorted(schemas):
    train_rows += sorted([r for r in approved if r['function_name']==fn and r['split']=='train'],key=lambda r:r['conversation_id'])[:MAX_TRAIN_PER_FUNCTION]
    eval_rows += sorted([r for r in approved if r['function_name']==fn and r['split']=='validation'],key=lambda r:r['conversation_id'])[:MAX_EVAL_PER_FUNCTION]
assert set(r['conversation_id'] for r in train_rows).isdisjoint(r['conversation_id'] for r in eval_rows)
APPROVAL_READY=True
print('파일럿 실제 수량',len(train_rows),len(eval_rows),'원본 split 유지')


## 2 GPU와 패키지
`RUN_MODEL_STEPS`는 파일럿 실행 설정입니다. 기본값 False이며 승인 검사를 통과한 후 변경합니다. 설치 조합과 GPU를 파일로 기록합니다. 첫 실행에서 설치 후 재시작 안내가 나오면 런타임을 다시 시작하고 처음부터 실행하세요.

In [ ]:
assert APPROVAL_READY,'데이터 승인 검사를 먼저 완료하세요'
assert RUN_MODEL_STEPS,'모델 크기를 확인한 후 설정에서 RUN_MODEL_STEPS=True로 변경하세요'
subprocess.check_call([sys.executable,'-m','pip','install','-U','transformers','peft','accelerate','bitsandbytes'])
import torch,transformers,peft,inspect,re,random
from huggingface_hub import HfApi
assert torch.cuda.is_available(),'GPU 런타임을 선택하세요'
random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR.mkdir(parents=True,exist_ok=False) # 기존 실험을 덮어쓰지 않음 새 실행은 새 경로 설정
REVISION=HfApi().model_info(MODEL_ID,revision=MODEL_REF).sha
assert re.fullmatch(r'[0-9a-f]{40}',REVISION)
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
versions={'torch':torch.__version__,'transformers':transformers.__version__,'peft':peft.__version__,'gpu':torch.cuda.get_device_name(0),'dtype':str(DTYPE),'model_id':MODEL_ID,'revision':REVISION,'source_sha256':helper.sha256(SOURCE),'review_sha256':helper.sha256(HUMAN),'reference_sha256':helper.sha256(REFERENCE),'seed':SEED,'train_count':len(train_rows),'eval_count':len(eval_rows)}
(OUTPUT_DIR/'environment.json').write_text(json.dumps(versions,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
print(versions)


## 3 49개 토큰 등록 및 정답 구간 검사
Qwen 기본 EOS를 유지하고 hs-end에서 별도로 중단합니다. 프롬프트에 포함된 빈 think 영역을 모델 출력으로 평가하지 않습니다. 전체 assistant 메시지를 템플릿에 재직렬화하지 않고 생성 프롬프트와 hs completion을 분리합니다.

In [ ]:
assert APPROVAL_READY and RUN_MODEL_STEPS
from transformers import AutoTokenizer
TOKEN_CHECKED=MASK_CHECKED=False
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION)
BASE_EOS=tokenizer.eos_token_id
assert BASE_EOS is not None and tokenizer.chat_template
params=inspect.signature(tokenizer.add_special_tokens).parameters
if 'replace_extra_special_tokens' in params:
    tokenizer.add_special_tokens({'extra_special_tokens':tokens},replace_extra_special_tokens=False)
else:
    tokenizer.add_special_tokens({'additional_special_tokens':tokens},replace_additional_special_tokens=False)
ids={s:tokenizer.encode(s,add_special_tokens=False) for s in tokens}
assert len(tokens)==49 and all(len(x)==1 for x in ids.values())
token_ids={s:v[0] for s,v in ids.items()}
assert len(set(token_ids.values()))==49 and BASE_EOS not in token_ids.values()
assert tokenizer.eos_token_id==BASE_EOS
for text,i in token_ids.items():assert tokenizer.decode([i],skip_special_tokens=False)==text
END_ID=token_ids['<hs-end>']
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
TOKEN_CHECKED=True
train_encoded=[helper.encode_case(tokenizer,r,system,MAX_LENGTH) for r in train_rows]
eval_encoded=[helper.encode_case(tokenizer,r,system,MAX_LENGTH) for r in eval_rows]
for x,r in zip(train_encoded+eval_encoded,train_rows+eval_rows):
    pos=next(i for i,v in enumerate(x['labels']) if v!=-100)
    assert all(v==-100 for v in x['labels'][:pos])
    assert x['labels'][pos:]==tokenizer.encode(r['llm_output'],add_special_tokens=False)
    assert x['labels'][-1]==END_ID and x['labels'][pos] in token_ids.values()
    assert x['input_ids'][pos:]==x['labels'][pos:]
MASK_CHECKED=True
(OUTPUT_DIR/'token_manifest.json').write_text(json.dumps({'tokens':token_ids,'base_eos':BASE_EOS,'model_id':MODEL_ID,'base_revision':REVISION,'system_sha256':hashlib.sha256(system.encode()).hexdigest()},ensure_ascii=False,indent=2),encoding='utf-8')
print('단일 토큰49 및 loss mask 확인',max(len(x['input_ids']) for x in train_encoded+eval_encoded))


## 4 모델 로딩과 LoRA
4bit는 GPU 메모리를 줄이는 파일럿 로딩 방식입니다. 보드용 양자화 결과가 아닙니다. 텍스트 영역의 실제 q_proj·v_proj만 선택하고 새 토큰 임베딩도 학습합니다. 선택 모델이나 라이브러리에서 지원하지 않으면 중단합니다.

In [ ]:
assert APPROVAL_READY and RUN_MODEL_STEPS and TOKEN_CHECKED and MASK_CHECKED
from transformers import BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel
MODEL_CLASS=getattr(transformers,'Qwen3_5ForConditionalGeneration',None)
assert MODEL_CLASS is not None,'현재 transformers가 Qwen3.5 full config를 지원하지 않습니다'
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=DTYPE)
def load_base():
    m=MODEL_CLASS.from_pretrained(MODEL_ID,revision=REVISION,quantization_config=quant,device_map={'':0},dtype=DTYPE,attn_implementation='sdpa')
    m.resize_token_embeddings(max(len(tokenizer),m.get_input_embeddings().num_embeddings))
    assert max(token_ids.values())<m.get_input_embeddings().num_embeddings
    assert m.get_input_embeddings().weight.data_ptr()==m.get_output_embeddings().weight.data_ptr(),'이 파일럿은 tied embedding 경로만 지원함'
    return m
base=load_base()
base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
embed_name=next(name for name,module in base.named_modules() if module is base.get_input_embeddings())
text_root=embed_name.rsplit('.',1)[0]
targets=[name for name,module in base.named_modules() if name.startswith(text_root+'.') and name.rsplit('.',1)[-1] in {'q_proj','v_proj'} and isinstance(module,torch.nn.Linear)]
assert targets and all('visual' not in name and 'vision' not in name for name in targets)
assert 'trainable_token_indices' in inspect.signature(LoraConfig).parameters and 'ensure_weight_tying' in inspect.signature(LoraConfig).parameters
config=LoraConfig(task_type='CAUSAL_LM',r=8,lora_alpha=16,lora_dropout=0.05,target_modules=targets,trainable_token_indices={embed_name:list(token_ids.values())},ensure_weight_tying=True)
model=get_peft_model(base,config)
model.config.use_cache=False
model.print_trainable_parameters()
print('텍스트 LoRA 대상',targets,'추가 토큰 대상',embed_name)


## 5 gradient 소량 확인과 학습
먼저 한 사례의 backward로 추가 토큰 gradient를 검사하고 지웁니다. 이 확인에서는 optimizer step을 수행하지 않습니다. 다음 Trainer 단계만 실제 파일럿 학습입니다.

In [ ]:
assert APPROVAL_READY and TOKEN_CHECKED and MASK_CHECKED and RUN_MODEL_STEPS
GRADIENT_CHECKED=False
class Rows(torch.utils.data.Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):return self.rows[i]
def collate(rows):
    longest=max(len(r['input_ids']) for r in rows)
    return {k:torch.tensor([r[k]+[(-100 if k=='labels' else tokenizer.pad_token_id if k=='input_ids' else 0)]*(longest-len(r[k])) for r in rows],dtype=torch.long) for k in ['input_ids','attention_mask','labels']}
model.train();device=model.get_input_embeddings().weight.device
batch={k:v.to(device) for k,v in collate([train_encoded[0]]).items()}
loss=model(**batch).loss
assert torch.isfinite(loss)
loss.backward()
token_grads=[p.grad for name,p in model.named_parameters() if 'trainable_tokens' in name and p.requires_grad]
assert token_grads and all(g is not None and torch.isfinite(g).all() for g in token_grads)
assert any(g.abs().sum().item()>0 for g in token_grads)
model.zero_grad(set_to_none=True);del batch,loss,token_grads
GRADIENT_CHECKED=True
from transformers import Trainer,TrainingArguments
args=TrainingArguments(output_dir=str(OUTPUT_DIR/'trainer'),max_steps=MAX_STEPS,per_device_train_batch_size=1,gradient_accumulation_steps=4,learning_rate=2e-4,bf16=DTYPE==torch.bfloat16,fp16=DTYPE==torch.float16,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},logging_steps=1,save_strategy='no',report_to='none',seed=SEED,remove_unused_columns=False,optim='adamw_torch')
trainer=Trainer(model=model,args=args,train_dataset=Rows(train_encoded),data_collator=collate)
assert GRADIENT_CHECKED
train_result=trainer.train()
assert trainer.state.global_step==MAX_STEPS
trainer.save_state()
(OUTPUT_DIR/'train_metrics.json').write_text(json.dumps(train_result.metrics,indent=2),encoding='utf-8')


## 6 검증과 저장·재로딩
기능·인자·전체 토큰·형식·종료를 별도로 집계합니다. 잘못된 출력은 실행하지 않습니다. handler나 TTS는 이 노트북에서 호출하지 않습니다. tokenizer와 adapter를 함께 저장하고 재로딩 결과를 대조합니다.

In [ ]:
assert APPROVAL_READY and RUN_MODEL_STEPS and GRADIENT_CHECKED
id_tokens={i:s for s,i in token_ids.items()}
def evaluate(m):
    m.eval();results=[]
    for row,x in zip(eval_rows,eval_encoded):
        pos=next(i for i,y in enumerate(x['labels']) if y!=-100)
        dev=m.get_input_embeddings().weight.device
        prompt=torch.tensor([x['input_ids'][:pos]],device=dev)
        with torch.no_grad():out=m.generate(input_ids=prompt,attention_mask=torch.ones_like(prompt),max_new_tokens=8,do_sample=False,eos_token_id=[END_ID,BASE_EOS],pad_token_id=tokenizer.pad_token_id,use_cache=True)[0,pos:].tolist()
        strings=[id_tokens.get(i,f'UNKNOWN_ID_{i}') for i in out]
        try:fn,parsed=helper.parse_output(strings,schemas);valid=True
        except ValueError:fn=None;parsed=None;valid=False
        results.append({'conversation_id':row['conversation_id'],'expected_function':row['function_name'],'output_ids':out,'format_valid':valid,'end_valid':bool(out and out[-1]==END_ID and out.count(END_ID)==1),'function_correct':fn==row['function_name'],'arguments_correct':valid and fn==row['function_name'] and parsed==json.loads(row['arguments_json']),'exact_correct':out==tokenizer.encode(row['llm_output'],add_special_tokens=False)})
    summary={k:sum(r[k] for r in results)/len(results) for k in ['format_valid','end_valid','function_correct','arguments_correct','exact_correct']}
    summary['per_function']={fn:{'support':sum(r['expected_function']==fn for r in results),'function_accuracy':sum(r['function_correct'] for r in results if r['expected_function']==fn)/sum(r['expected_function']==fn for r in results)} for fn in schemas}
    summary['macro_function_accuracy']=sum(v['function_accuracy'] for v in summary['per_function'].values())/len(schemas)
    return results,summary
before_reload,metrics=evaluate(model)
(OUTPUT_DIR/'eval_cases.json').write_text(json.dumps(before_reload,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'eval_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
ADAPTER_DIR=OUTPUT_DIR/'adapter'
model.save_pretrained(ADAPTER_DIR,safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)
del trainer,model,base
torch.cuda.empty_cache()
loaded_tok=AutoTokenizer.from_pretrained(ADAPTER_DIR)
assert {s:loaded_tok.encode(s,add_special_tokens=False)[0] for s in tokens}==token_ids
assert all(len(loaded_tok.encode(s,add_special_tokens=False))==1 for s in tokens)
assert loaded_tok.eos_token_id==BASE_EOS and loaded_tok.chat_template==tokenizer.chat_template
assert [helper.encode_case(loaded_tok,r,system,MAX_LENGTH) for r in eval_rows]==eval_encoded
reloaded_base=load_base()
reloaded=PeftModel.from_pretrained(reloaded_base,ADAPTER_DIR,is_trainable=False)
after_reload,_=evaluate(reloaded)
assert [r['output_ids'] for r in before_reload]==[r['output_ids'] for r in after_reload],'재로딩 결과가 다릅니다 저장 형식과 추가 토큰 가중치를 확인하세요'
(OUTPUT_DIR/'reload_check.json').write_text(json.dumps({'token_ids_equal':True,'greedy_outputs_equal':True,'validation_cases':len(after_reload),'quantization':'4bit Colab loading only not board export'}),encoding='utf-8')
print(metrics)
print('저장 위치',OUTPUT_DIR)


## 범위와 참고
이 노트북은 소량 파일럿입니다. 현재 131개 검수표만으로 승인 2800개를 확보한 것이 아닙니다. 별도 승인 validation만 평가하며 이를 학습·양자화 calibration에 사용하지 않습니다. 실제 STT 음성 평가, TTS 음성 재생, 보드용 양자화와 handler 통합은 별도 작업입니다.

공식 자료: [Qwen3.5 Transformers](https://huggingface.co/docs/transformers/model_doc/qwen3_5) · [Qwen chat template](https://huggingface.co/Qwen/Qwen3.5-2B/blob/main/chat_template.jinja) · [PEFT LoRA](https://huggingface.co/docs/peft/main/package_reference/lora)
